# pragmatic-dimensions-classifier — Phase 1: Data Familiarization & Preliminary Baselines

**Status:** Exploratory. See `TASK.md` for the full brief.

This notebook loads two open formality datasets — Pavlick & Tetreault (2016) and SQUINKY! (Lahiri 2015) — inspects their structure, and runs a few preliminary classifier tests. The goal is to get hands-on with both corpora and surface concrete signal (especially: how hard is cross-corpus generalization?) before finalizing the design of the actual formality/register classifier project.

**Not the final design.** Binarization thresholds, model choice, and preprocessing here are deliberately simple — good enough to see signal, not good enough to ship.

## 0. Setup

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Load Pavlick & Tetreault (2016)

License: CC-BY-3.0. Columns: `domain` (news / blog / email / answers), `avg_score` (float, −3 to 3, lower = less formal), `sentence`. See `data/README.md` for citation requirements (cite both Pavlick & Tetreault 2016 and Lahiri 2015).

In [ ]:
pt_dataset = load_dataset("osyvokon/pavlick-formality-scores")
pt_train = pt_dataset["train"].to_pandas()
pt_test = pt_dataset["test"].to_pandas()
pt_all = pd.concat([pt_train, pt_test], ignore_index=True)

print(pt_all.shape)
print(pt_all["domain"].value_counts())
pt_all.head()

### 1.1 Inspect distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(pt_all["avg_score"], bins=30, ax=axes[0])
axes[0].set_title("Pavlick-Tetreault: formality score distribution (all genres)")

sns.boxplot(data=pt_all, x="domain", y="avg_score", ax=axes[1])
axes[1].set_title("Formality by genre")
plt.tight_layout()
plt.show()

pt_all.groupby("domain")["avg_score"].describe()

✍️ **Your observations:** _(genre patterns, anything surprising, anything that affects later design)_

- Clear genre hierarchy in formality: news (mean 0.72) > email (0.46) > blog (0.21) > answers (−0.73). The ranking is intuitive: newswire prose sits firmly in the formal register, while answers (likely Q&A forum text, e.g. Yahoo Answers) is the only genre with a negative mean, placing it solidly in the informal range.

- Answers dominates the corpus (4,977 of 11,274 sentences, ~44%) and is the most informal genre by a large margin. This creates a mild class imbalance concern after binarization: depending on the threshold chosen, the informal class will be disproportionately drawn from a single genre, which could inflate cross-genre error rates.

- Genre differences in variance: news has the tightest distribution (std 0.86) — newswire text is stylistically uniform. Email and answers show considerably more spread (std ~1.3–1.4), reflecting the wider range of informal register variation. This heterogeneity within the informal genres may make them harder to classify consistently.

- Scale note for cross-corpus comparison: PT uses a continuous −3 to 3 scale (crowd-averaged), while SQUINKY! uses 1–7. The PT distribution is also more symmetric around zero, whereas SQUINKY! formality is slightly right-skewed (mean 3.91 on a 1–7 scale, i.e. just below the midpoint). Any cross-corpus experiment will need to account for this scale difference — either via normalization or by treating binarization thresholds independently per corpus.

## 2. Load SQUINKY! (Lahiri 2015)

**Column structure is not fully documented** — see `TASK.md` for what's known from the reference implementation (`meyersbs/squinky`). Inspect before assuming anything. The raw file is hosted on a personal academic homepage, not a stable archive — if the download below fails, check the GitHub repo's issues for a mirror.

In [ ]:
import urllib3
# The RIT server redirects HTTP -> HTTPS but has a broken certificate chain (missing intermediate).
# verify=False is acceptable here: this is a known academic dataset from a known URL,
# not a security-sensitive operation. The warning is suppressed to avoid log noise.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

SQUINKY_URL = "http://people.rc.rit.edu/~bsm9339/corpora/squinky_corpus/mturk_merged.csv"
SQUINKY_PATH = os.path.join(DATA_DIR, "squinky_mturk_merged.csv")

if not os.path.exists(SQUINKY_PATH):
    resp = requests.get(SQUINKY_URL, timeout=30, verify=False)
    resp.raise_for_status()
    with open(SQUINKY_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded to {SQUINKY_PATH}")
else:
    print("Already downloaded.")

### 2.1 Inspect raw structure before assuming anything

In [ ]:
squinky_raw = pd.read_csv(SQUINKY_PATH)
print(squinky_raw.shape)
print(squinky_raw.columns.tolist())

print(squinky_raw.iloc[0]['sentence'])

print(squinky_raw.shape)
print(squinky_raw.dtypes)
squinky_raw.iloc[0]   # all column values for row 0

squinky_raw.head()


**Note:** if the printed columns above aren't self-explanatory, rename them based on position, following the convention used in the reference implementation (`meyersbs/squinky/squinky/classifier.py`): column 0 = id, column 1 = formality (1–7), column 2 = informativeness (1–7), column 3 = implicature (1–7), last column = sentence text. There may be additional columns (e.g. genre/source) in between that aren't documented anywhere — confirm against the printed output above, don't assume.

### 2.1b Reproducibility checks

Two validation tests before committing to the cleaned `squinky` dataframe:

**Test 1 — Classifier reproducibility.** The `meyersbs/squinky` reference implementation reports macro F1 of 0.82 / 0.84 / 0.60 for formality / informativeness / implicature using TF-IDF + a classifier with binarization at score > 4. We replicate that here to verify the data loaded correctly and the binarization convention works.

**Test 2 — Genre assignment by ID range.** The paper's Table 1 (Spearman ρ between two MTurk rounds) *cannot* be reproduced from the merged dataset — we only have the averaged ratings, not the individual round means. Instead, we validate the genre assignment hypothesis (blog IDs 0–2109, news 2110–5118, forum 5119–7031) against the inter-dimension Spearman correlations from Table 3 of the paper, which we can compute directly.

In [ ]:
from scipy.stats import spearmanr

dims = ["formality", "informativeness", "implicature"]

# ── Test 1: Classifier reproducibility ────────────────────────────────────────
# Reference (meyersbs/squinky): Formality 0.82, Informativeness 0.84, Implicature 0.60
# Binarization: score > 4 → positive class (reference implementation convention)
ref_f1 = {"formality": 0.82, "informativeness": 0.84, "implicature": 0.60}

print("=== Test 1: Classifier reproducibility (TF-IDF + LogReg, score > 4) ===\n")
for dim in dims:
    labels = (squinky_raw[dim] > 4).astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        squinky_raw["sentence"], labels, test_size=0.2, random_state=42, stratify=labels
    )
    vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    clf = LogisticRegression(max_iter=1000)
    clf.fit(vec.fit_transform(X_tr), y_tr)
    preds = clf.predict(vec.transform(X_te))
    macro_f1 = classification_report(y_te, preds, output_dict=True)["macro avg"]["f1-score"]
    match = "✓" if abs(macro_f1 - ref_f1[dim]) <= 0.03 else "✗"
    print(f"  {dim.capitalize():16s}  macro F1 = {macro_f1:.2f}  (reference: {ref_f1[dim]:.2f})  {match}")
    print(classification_report(y_te, preds, digits=2))

# ── Test 2: Genre assignment by ID range ──────────────────────────────────────
# Paper corpus composition (Lahiri 2015, Section 2.1): blog=2110, news=3009, forum=2569
# Hypothesis: rows are stored in that order in the CSV
print("\n=== Test 2: Genre assignment by ID range ===\n")
squinky_raw = squinky_raw.copy()
squinky_raw["genre"] = pd.cut(
    squinky_raw["id"],
    bins=[-1, 2109, 5118, 7031],
    labels=["blog", "news", "forum"]
)
print(f"Genre counts: {squinky_raw['genre'].value_counts().to_dict()}")
print("\nGenre-wise means (formality should rank news > blog > forum per paper):")
print(squinky_raw.groupby("genre", observed=True)[dims].mean().round(3))

# Validate against Table 3 (Lahiri 2015): inter-dimension Spearman ρ per genre
# Paper values: Overall Fo-In=0.73, Fo-Im=0.07, In-Im=0.05
#               Blog    Fo-In=0.73, Fo-Im=-0.10, In-Im=-0.08
#               News    Fo-In=0.63, Fo-Im=-0.08, In-Im=-0.10
#               Forum   Fo-In=0.57, Fo-Im=0.04,  In-Im=0.08
paper_t3 = {
    "overall": (0.73,  0.07,  0.05),
    "blog":    (0.73, -0.10, -0.08),
    "news":    (0.63, -0.08, -0.10),
    "forum":   (0.57,  0.04,  0.08),
}
print("\nSpearman ρ between dimensions — ours vs. paper Table 3:")
print(f"  {'Genre':8s}  {'Fo–In':>14}  {'Fo–Im':>14}  {'In–Im':>14}")
for genre in ["overall", "blog", "news", "forum"]:
    sub = squinky_raw if genre == "overall" else squinky_raw[squinky_raw["genre"] == genre]
    fo_in = spearmanr(sub["formality"], sub["informativeness"]).statistic
    fo_im = spearmanr(sub["formality"], sub["implicature"]).statistic
    in_im = spearmanr(sub["informativeness"], sub["implicature"]).statistic
    pv = paper_t3[genre]
    print(f"  {genre:8s}  {fo_in:+.2f} [{pv[0]:+.2f}]  {fo_im:+.2f} [{pv[1]:+.2f}]  {in_im:+.2f} [{pv[2]:+.2f}]")

In [ ]:
squinky = squinky_raw.copy()

# Strip leading digits that appear to be corpus-internal sentence IDs
# concatenated without a space in the original data (data quality issue, not a parsing artifact)
squinky["sentence"] = squinky["sentence"].str.replace(r"^\d+", "", regex=True).str.lstrip()

squinky.head()

### 2.2 Inspect distributions (formality, informativeness, implicature)

In [ ]:
dims = ["formality", "informativeness", "implicature"]

# Overall score distributions for all three dimensions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, dim in zip(axes, dims):
    sns.histplot(squinky[dim], bins=20, ax=ax)
    ax.set_title(f"SQUINKY!: {dim} distribution")
    ax.set_xlabel("Score (1–7)")
plt.suptitle("SQUINKY! — score distributions (all sentences)", y=1.02)
plt.tight_layout()
plt.show()

# Pairwise relationships between dimensions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [("formality", "informativeness"), ("formality", "implicature"), ("informativeness", "implicature")]
for ax, (x, y) in zip(axes, pairs):
    ax.scatter(squinky[x], squinky[y], alpha=0.15, s=10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} vs {y} (r={squinky[x].corr(squinky[y]):.2f})")
plt.suptitle("SQUINKY! — pairwise correlations between dimensions", y=1.02)
plt.tight_layout()
plt.show()

# Summary statistics
squinky[dims].describe()

✍️ **Your observations:** _(how does the 1–7 scale compare to PT's −3..3? any class imbalance? does implicature look noisier, as the paper warns?)_

- Formality (mean 3.91, std 1.12) and informativeness (mean 4.48, std 1.05) show similar spread across the 1–7 scale and are strongly correlated (r=0.77). This is linguistically expected: formal registers (news, academic prose) tend to pack more propositional content, while informal registers are more phatic and contextually anchored. For modeling, the two dimensions are not independent — a classifier trained on formality features will partly be learning informativeness signal too.

- Implicature (mean 4.00, std 0.67) is uncorrelated with both formality (r=0.01) and informativeness (r=0.09), consistent with Lahiri (2015) and theoretically sensible — implicature is orthogonal to register. Importantly, the distribution is not noisier than the others; it is in fact tighter (lowest std). The challenge for classification is low variance rather than label noise: most sentences cluster near the midpoint, leaving little discriminative signal above/below any binarization threshold. Implicature is therefore less suited as a primary classification target but interesting as a secondary analysis ("what can surface form even detect?").

- No genre column exists in SQUINKY! — unlike Pavlick-Tetreault, there is no domain-level breakdown available for stratified analysis. This is a meaningful structural difference between the two corpora and a relevant constraint for cross-corpus generalization experiments.

## 3. Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({
    "Pavlick-Tetreault (2016)": {
        "N sentences": len(pt_all),
        "Dimensions": "Formality",
        "Scale": "-3 to 3 (continuous)",
        "Genres": ", ".join(sorted(pt_all["domain"].unique())),
        "License": "CC-BY-3.0",
    },
    "SQUINKY! (Lahiri 2015)": {
        "N sentences": len(squinky),
        "Dimensions": "Formality, Informativeness, Implicature",
        "Scale": "1 to 7 (continuous)",
        "Genres": "news, blog, forum (per paper — confirm against data)",
        "License": "Code: MIT; data license unconfirmed",
    },
})
comparison

## 4. Preliminary classifier tests

Three quick tests, in increasing order of interest:

- **Model A** — TF-IDF + Logistic Regression, in-domain on Pavlick-Tetreault.
- **Model B** — TF-IDF + Logistic Regression, in-domain on SQUINKY! formality.
- **Model C** — cross-corpus: train on one, evaluate on the other. This is the diagnostic that actually matters for the final design — it tells us how viable "train on one corpus, evaluate cross-corpus on the other" is as a project framing, before we commit to it.

Binarization below is a simple median/threshold split, chosen for speed — **not** the final design decision. That's flagged explicitly at each step.

In [ ]:
def run_tfidf_logreg(train_texts, train_labels, test_texts, test_labels, label="Model"):
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, train_labels)
    preds = clf.predict(X_test)

    print(f"--- {label} ---")
    print(classification_report(test_labels, preds))
    print("Confusion matrix:")
    print(confusion_matrix(test_labels, preds))
    return clf, vectorizer, preds

### Model A — Pavlick-Tetreault, in-domain

In [ ]:
# Binarization choice (preliminary only — revisit in the design discussion):
# median split, since avg_score has no natural zero-formality cutoff across genres.
pt_threshold = pt_all["avg_score"].median()
pt_all["formal_label"] = (pt_all["avg_score"] > pt_threshold).astype(int)
print(f"PT median threshold: {pt_threshold}")
print(pt_all["formal_label"].value_counts())

pt_train_texts, pt_test_texts, pt_train_labels, pt_test_labels = train_test_split(
    pt_all["sentence"], pt_all["formal_label"],
    test_size=0.2, random_state=42, stratify=pt_all["formal_label"]
)

model_a, vec_a, preds_a = run_tfidf_logreg(
    pt_train_texts, pt_train_labels, pt_test_texts, pt_test_labels,
    label="Model A: Pavlick-Tetreault in-domain"
)

### Model B — SQUINKY!, in-domain

In [ ]:
# TODO: mirror Model A once SQUINKY! columns are confirmed/renamed above.
# Suggested binarization to start with (matches the reference implementation's
# convention, NOT necessarily what we'll use in the final project):

squinky["formal_label"] = (squinky["formality"] > 4).astype(int)

sq_train_texts, sq_test_texts, sq_train_labels, sq_test_labels = train_test_split(
    squinky["sentence"], squinky["formal_label"],
    test_size=0.2, random_state=42, stratify=squinky["formal_label"]
)

model_b, vec_b, preds_b = run_tfidf_logreg(
    sq_train_texts, sq_train_labels, sq_test_texts, sq_test_labels,
    label="Model B: SQUINKY! in-domain"
)

### Model C — Cross-corpus generalization (the interesting one)

Train on one corpus, evaluate on the other. A large F1 drop relative to the in-domain models is expected — the question is *how* large, and whether it looks genre-driven, scale-driven, or both. This is the central diagnostic for whether "train on one, evaluate cross-corpus on the other" is viable as the eventual project framing, or whether it needs more careful harmonization first (rescaling, genre-matched subsets, vocabulary normalization).

In [ ]:
# TODO: once Model B's labels exist, run both directions:
#
# Direction 1 — train on PT, evaluate on SQUINKY!:
vec_c1 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_pt = vec_c1.fit_transform(pt_all["sentence"])
clf_c1 = LogisticRegression(max_iter=1000).fit(X_train_pt, pt_all["formal_label"])
X_test_squinky = vec_c1.transform(squinky["sentence"])
preds_c1 = clf_c1.predict(X_test_squinky)
print("--- Model C1: train PT -> test SQUINKY! ---")
print(classification_report(squinky["formal_label"], preds_c1))

# Direction 2 — train on SQUINKY!, evaluate on PT (repeat symmetrically):
vec_c2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_sq = vec_c2.fit_transform(squinky["sentence"])
clf_c2 = LogisticRegression(max_iter=1000).fit(X_train_sq, squinky["formal_label"])
X_test_pt = vec_c2.transform(pt_all["sentence"])
preds_c2 = clf_c2.predict(X_test_pt)
print("--- Model C2: train SQUINKY! -> test PT ---")
print(classification_report(pt_all["formal_label"], preds_c2))

# Compare both directions' F1 against the in-domain Model A/B scores above.

✍️ **Your observations:** _(how much does cross-corpus accuracy drop? genre-driven or scale-driven? does this support the cross-corpus framing, or suggest a different angle?)_

Cross-corpus F1 (C1: 0.79, C2: 0.75) matches or exceeds in-domain performance (Model A: 0.76, Model B: 0.81), suggesting strong cross-corpus transfer of surface-level formality signal. The asymmetry in recall — both directions show higher recall for the informal class — points to informal vocabulary (forum/answers text) being more distinctively lexical and therefore more transferable than formal register markers. The absence of a significant generalization gap is a meaningful positive result for the project framing, though genre-stratified evaluation would be needed to confirm the effect is not driven by easy genre-extreme examples in both corpora.

### Model C, genre-stratified — is the transfer real or a register-extremes artifact? (Open Question 1)

The aggregate cross-corpus F1 (C2: 0.75) could be flattered by easy genre-extreme sentences. Since PT has a `domain` column, we can split the C2 predictions (train SQUINKY! → test PT) by genre and check whether performance is uniform.

**What we're looking for:**
- If macro F1 stays in a similar range (within ~0.10–0.15) across news, email, blog, and answers → the "strong cross-corpus transfer" story holds.
- If extreme registers (news, answers) score well while mid-register genres (blog, email) collapse toward the per-genre majority-class baseline → the aggregate number was driven by easy extremes, and the honest framing becomes "the classifier separates extreme registers; mid-register generalization is unresolved."

Only the C2 direction (PT as test set) can be stratified this way — SQUINKY! has no genre labels, so C1 cannot be broken down. We reuse the already-trained `clf_c2` / `vec_c2` and the `preds_c2` produced above (which align row-for-row with `pt_all`).

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# preds_c2 was produced above as clf_c2.predict(vec_c2.transform(pt_all["sentence"])),
# so it aligns row-for-row with pt_all. We attach it and stratify by PT's domain.
pt_eval = pt_all[["domain", "formal_label"]].copy()
pt_eval["pred"] = preds_c2

# Order domains from most formal to least formal (per the PT distribution means)
domain_order = ["news", "email", "blog", "answers"]

rows = []
for domain in domain_order + ["OVERALL"]:
    sub = pt_eval if domain == "OVERALL" else pt_eval[pt_eval["domain"] == domain]
    y, p = sub["formal_label"], sub["pred"]
    # Majority-class baseline macro F1 for this subset — the "collapse toward chance" reference
    majority = int(y.mean() >= 0.5)
    baseline_f1 = f1_score(y, [majority] * len(y), average="macro")
    rows.append({
        "domain": domain,
        "n": len(sub),
        "%_formal": round(100 * y.mean(), 1),
        "accuracy": round(accuracy_score(y, p), 3),
        "macro_F1": round(f1_score(y, p, average="macro"), 3),
        "baseline_F1": round(baseline_f1, 3),
        "F1_informal": round(f1_score(y, p, pos_label=0), 3),
        "F1_formal": round(f1_score(y, p, pos_label=1), 3),
    })

genre_breakdown = pd.DataFrame(rows).set_index("domain")
print("Model C2 (train SQUINKY! -> test PT), stratified by PT domain:\n")
print(genre_breakdown)

# Visualize per-genre macro F1 against the majority-class baseline and the overall score
dom = genre_breakdown.drop("OVERALL")
x = np.arange(len(dom))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - 0.2, dom["macro_F1"], width=0.4, label="model macro F1", color="steelblue")
ax.bar(x + 0.2, dom["baseline_F1"], width=0.4, label="majority-class baseline", color="lightgray")
ax.axhline(genre_breakdown.loc["OVERALL", "macro_F1"], ls="--", color="black",
           label=f"overall macro F1 ({genre_breakdown.loc['OVERALL', 'macro_F1']:.2f})")
ax.set_xticks(x)
ax.set_xticklabels(dom.index)
ax.set_ylabel("macro F1")
ax.set_ylim(0, 1)
ax.set_title("Cross-corpus transfer (SQUINKY! → PT): macro F1 by PT genre")
ax.legend()
plt.tight_layout()
plt.show()

### Confirmatory test — is the per-genre weakness a threshold issue?

The breakdown above suggests the residual cost is class-imbalance/threshold mismatch, not a generalization gap: within each genre the model is weak on the locally-minority class. If that diagnosis is right, retraining C2 with `class_weight="balanced"` (which stops the model from defaulting to the dominant class) should lift the minority-class F1 — especially `F1_formal` in `answers` and `F1_informal` in `news` — and raise the genres that were dragged down by imbalance.

If instead the balanced model barely changes, the weakness would be a genuine signal limitation rather than a thresholding artifact.

In [ ]:
# Reuse the C2 vectorizer (vec_c2) and PT test matrix (X_test_pt) from the Model C cell.
# Only the classifier changes: class_weight="balanced" reweights the loss by inverse
# class frequency so the model can't coast on the locally-dominant class.
clf_c2_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_c2_bal.fit(vec_c2.transform(squinky["sentence"]), squinky["formal_label"])
preds_c2_bal = clf_c2_bal.predict(X_test_pt)

domain_order = ["news", "email", "blog", "answers"]
pt_eval_bal = pt_all[["domain", "formal_label"]].copy()
pt_eval_bal["pred"] = preds_c2_bal

rows = []
for domain in domain_order + ["OVERALL"]:
    sub = pt_eval_bal if domain == "OVERALL" else pt_eval_bal[pt_eval_bal["domain"] == domain]
    y, p = sub["formal_label"], sub["pred"]
    rows.append({
        "domain": domain,
        "n": len(sub),
        "%_formal": round(100 * y.mean(), 1),
        "accuracy": round(accuracy_score(y, p), 3),
        "macro_F1": round(f1_score(y, p, average="macro"), 3),
        "F1_informal": round(f1_score(y, p, pos_label=0), 3),
        "F1_formal": round(f1_score(y, p, pos_label=1), 3),
    })
genre_breakdown_bal = pd.DataFrame(rows).set_index("domain")
print("Model C2 with class_weight='balanced' (train SQUINKY! -> test PT), by PT domain:\n")
print(genre_breakdown_bal)

# Side-by-side macro F1: default vs. balanced (genre_breakdown is from the previous cell)
compare = pd.DataFrame({
    "macro_F1 (default)": genre_breakdown["macro_F1"],
    "macro_F1 (balanced)": genre_breakdown_bal["macro_F1"],
})
compare["delta"] = (compare["macro_F1 (balanced)"] - compare["macro_F1 (default)"]).round(3)
print("\nComparison (default vs. balanced):")
print(compare)

### Threshold sweep — is the minority-class weakness a threshold location issue?

`class_weight='balanced'` barely moved the numbers, which means the training signal is not the problem. The hypothesis shifts: the model's *probability rankings* are correct, but the fixed 0.5 inference threshold is misaligned with each genre's class distribution.

To test this, we sweep the decision threshold from 0.1 to 0.9 and find the macro-F1-maximising threshold per genre. If the optimal threshold varies substantially across genres (and if minority-class F1 recovers at the per-genre optimum), the problem is threshold calibration — a harmonization issue, not a feature or generalisation gap. That directly motivates Questions 2 and 7 (binarization strategy, scale harmonisation) as the priority for Phase 2.

In [ ]:
from sklearn.metrics import roc_auc_score

thresholds = np.arange(0.1, 0.91, 0.02)
domain_order = ["news", "email", "blog", "answers"]

# Predicted probabilities from the default C2 classifier (no class_weight)
proba_c2 = clf_c2.predict_proba(X_test_pt)[:, 1]   # P(formal)
pt_eval_thresh = pt_all[["domain", "formal_label"]].copy()
pt_eval_thresh["proba"] = proba_c2

# For each genre: sweep threshold → find macro-F1-maximising threshold
results = {}
for domain in domain_order:
    sub = pt_eval_thresh[pt_eval_thresh["domain"] == domain]
    y = sub["formal_label"].values
    p = sub["proba"].values
    best_t, best_f1 = 0.5, 0.0
    f1_at_default = f1_score(y, (p >= 0.5).astype(int), average="macro")
    for t in thresholds:
        mf1 = f1_score(y, (p >= t).astype(int), average="macro")
        if mf1 > best_f1:
            best_f1, best_t = mf1, t
    results[domain] = {
        "%_formal": round(100 * y.mean(), 1),
        "macro_F1 @0.5": round(f1_at_default, 3),
        "best_threshold": round(best_t, 2),
        "macro_F1 @best": round(best_f1, 3),
        "gain": round(best_f1 - f1_at_default, 3),
        "AUC": round(roc_auc_score(y, p), 3),
    }

sweep_df = pd.DataFrame(results).T
sweep_df.index.name = "domain"
print("Threshold sweep results (C2: train SQUINKY! → test PT):\n")
print(sweep_df)

# Visualise: ROC curve per genre
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: F1 vs threshold per genre
ax = axes[0]
for domain in domain_order:
    sub = pt_eval_thresh[pt_eval_thresh["domain"] == domain]
    y, p = sub["formal_label"].values, sub["proba"].values
    f1s = [f1_score(y, (p >= t).astype(int), average="macro") for t in thresholds]
    ax.plot(thresholds, f1s, label=domain)
ax.axvline(0.5, ls="--", color="grey", lw=0.8, label="default (0.5)")
ax.set_xlabel("decision threshold")
ax.set_ylabel("macro F1")
ax.set_title("Macro F1 vs threshold by PT genre")
ax.legend()

# Right: per-genre macro F1 at default vs best threshold
ax = axes[1]
x = np.arange(len(domain_order))
ax.bar(x - 0.2, sweep_df["macro_F1 @0.5"], width=0.4, label="@0.5 (default)", color="steelblue")
ax.bar(x + 0.2, sweep_df["macro_F1 @best"], width=0.4, label="@best threshold", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(domain_order)
ax.set_ylabel("macro F1")
ax.set_ylim(0, 1)
ax.set_title("Per-genre gain from threshold tuning")
ax.legend()

plt.tight_layout()
plt.show()

## 5. Open questions for the design discussion

1. What's driving the cross-corpus transfer — register extremes or genuine generalization? Model C performs as well cross-corpus as in-domain, but both corpora contain genre-extreme sentences (newswire vs. forum/answers) with very different vocabularies. The F1 numbers may be flattering if the classifier is just pattern-matching on those extremes and struggling on mid-register sentences in both corpora. A genre-stratified evaluation on PT (which has domain labels) would test this. Decision: is the cross-corpus framing already validated, or does it need this check first?

2. Binarization strategy: median split vs. fixed threshold vs. continuous. Model A uses a median split (threshold = 0.0 for PT), Model B uses >4 (reference convention for SQUINKY!). These are not equivalent — the median split by definition gives balanced classes regardless of the actual score distribution, while >4 reflects a substantive claim about what "formal" means. For the final design: should both corpora use a consistent substantive threshold, or is per-corpus calibration acceptable? And is classification even the right framing, or should formality be treated as a regression target throughout?

3. Informativeness as a co-target or control variable. Formality and informativeness correlate at r=0.77 in SQUINKY!. A formality classifier will inevitably partially learn informativeness signal. Decision: should informativeness be included as a joint target, used as a feature, controlled for in the error analysis, or simply reported as a known confound?

4. Implicature: in or out? The low variance (std=0.67), central tendency bias in annotations, and the reference classifier's F1 of only 0.60 all point in the same direction. Decision: exclude from the main classifier pipeline and treat as a qualitative bonus analysis, or frame it explicitly as an upper-bound question ("what can surface form detect about implicature at all?")?

5. Genre information for SQUINKY!: worth pursuing? Genre labels are not in mturk_merged.csv and cannot be recovered by ID-range assignment. The paper's Google Drive link may have a richer version of the data. Decision: is it worth investigating before the final design, or is operating without genre stratification on SQUINKY! acceptable given that PT provides the genre-labeled half of the cross-corpus pair?

6. Transformer baseline: how much headroom is there? TF-IDF + LogReg gives 0.76–0.81 in-domain. Before committing to a transformer fine-tuning stage, it's worth knowing roughly how much improvement is realistic. Does the bottleneck look lexical (suggesting embeddings would help) or structural (suggesting the feature-engineering approach in the reference classifier is closer to the ceiling)?

## 6. Next steps

Bring the observations and open questions above into the design discussion. From there: lock in the binarization/regression approach, decide the final cross-corpus architecture (if pursued), and move on to the transformer fine-tuning stage (fast.ai Lesson 4 as the primary reference).